In [ ]:
import numpy as np
import pandas as pd
import optuna
import plotly
import joblib
import os
import json
import warnings
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import precision_recall_fscore_support, mean_squared_error
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from xgboost import XGBClassifier
from sentence_transformers import SentenceTransformer
from setfit import SetFitModel, SetFitTrainer, Trainer, TrainingArguments
from datasets import Dataset
from transformers import IntervalStrategy, EarlyStoppingCallback
from nltk.corpus import stopwords
from itertools import product

warnings.filterwarnings("ignore")
# ============================
# CONFIG
# ============================
DATA_PATH = r"Z:\PS_classification\df_curated.csv"
MODEL_PATH = r"Z:\Models\Modernbert_S_nl"

N_OUTER = 5
N_INNER = 5
N_TRIALS = 100
RANDOM_STATE = 42
feature_types=["tfidf","st","hybrid","setfit"]
model_types=["lr","xgb"] #"rf"


# ============================
# LOAD DATA
# ============================
df = pd.read_csv(DATA_PATH)
df.rename(columns={'censored':'text','curated_ps':'label'}, inplace=True)

df["has_ps"] = (df["label"] >= 0).astype(int)


# ============================
# EMBEDDINGS (FROZEN)
# ============================
try: 
    len(X_embed_all)
    print("Embeddings already computed")
except:
    print("Computing embeddings...")
    embed_model = SentenceTransformer(MODEL_PATH)
    df["embedding"] = list(embed_model.encode(df["text"].tolist()))
    X_embed_all = np.vstack(df["embedding"].values)

# ============================
# SETFIT
#=============================
args= TrainingArguments(batch_size=1, #32,
    num_epochs=5,
    load_best_model_at_end=True,
    body_learning_rate=2e-5,
    #head_learning_rate=5e-2,
    eval_strategy=IntervalStrategy.STEPS,
    eval_steps=50, 
    sampling_strategy='unique')
    #metric_for_best_model='f1')
new_context_length=4096

# ============================
# UTILS
# ============================
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def custom_adjacent_accuracy(y_true,y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return np.sum(np.abs(y_pred - y_true) <= 1) / len(y_pred)
    
def find_best_threshold(y_true, probs):
    thresholds = np.linspace(0.1, 0.9, 50)
    best_t = 0.5
    best_score = -np.inf

    for t in thresholds:
        preds = (probs >= t).astype(int)
        fp = ((preds == 1) & (y_true == 0)).sum()
        tp = ((preds == 1) & (y_true == 1)).sum()

        score = tp - 3 * fp  # penalize FP strongly

        if score > best_score:
            best_score = score
            best_t = t

    return best_t
    
def get_setfit_embeddings(model, texts, path):
    if os.path.exists(path):
        return np.load(path)
    else:
        emb = np.vstack(model.model_body.encode(texts,batch_size=8))
        np.save(path, emb)
        return emb
        
def cascade_predict(bin_model, mc_model, X_bin, X_mc, threshold=None):

    if hasattr(bin_model, "predict_proba") and threshold is not None:
        probs = bin_model.predict_proba(X_bin)[:,1]
        bin_pred = (probs >= threshold).astype(int)
    else:
        bin_pred = bin_model.predict(X_bin)

    mc_pred = mc_model.predict(X_mc) #X_bin

    final = [-1 if bin_pred[i]==0 else mc_pred[i] for i in range(len(bin_pred))]
    return np.array(final)



def get_model(trial, prefix=""): 


    model_type = trial.suggest_categorical(prefix+"model", ["lr","xgb"]) #,"rf",

    if model_type == "lr":
        return LogisticRegression(
            C=trial.suggest_float(prefix+"C", 1e-2, 10, log=True),
            max_iter=1000,
            random_state=RANDOM_STATE
        )

    elif model_type == "rf":
        return RandomForestClassifier(
            n_estimators=trial.suggest_int(prefix+"n", 50, 200),
            max_depth=trial.suggest_int(prefix+"depth", 3, 10),
            random_state=RANDOM_STATE
        )

    else:
        return XGBClassifier(
            n_estimators=trial.suggest_int(prefix+"n", 50, 300),
            max_depth=trial.suggest_int(prefix+"depth", 3, 10),
            learning_rate=trial.suggest_float(prefix+"lr", 0.01, 0.3, log=True),
            subsample=trial.suggest_float(prefix+"subsample", 0.6, 1.0),
            colsample_bytree=trial.suggest_float(prefix+"colsample", 0.6, 1.0),
            reg_lambda= trial.suggest_float(prefix+"lambda", 1e-3, 10, log=True), #1.5, #
            reg_alpha=  trial.suggest_float(prefix+"alpha", 1e-3, 10, log=True), #2.0,
            use_label_encoder=False,
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
def get_model_basic(trial, mod, prefix=""): 


    model_type = mod

    if model_type == "lr":
        return LogisticRegression(
           # C=trial.suggest_float(prefix+"C", 1e-2, 10, log=True),
            max_iter=1000,
            random_state=RANDOM_STATE
        )

    elif model_type == "rf":
        return RandomForestClassifier(
            #n_estimators=trial.suggest_int(prefix+"n", 50, 200),
            #max_depth=trial.suggest_int(prefix+"depth", 3, 10),
            random_state=RANDOM_STATE
        )

    else:
        return XGBClassifier(
            #n_estimators=trial.suggest_int(prefix+"n", 50, 300),
            #max_depth=trial.suggest_int(prefix+"depth", 3, 10),
            #learning_rate=trial.suggest_float(prefix+"lr", 0.01, 0.3, log=True),
            #subsample=trial.suggest_float(prefix+"subsample", 0.6, 1.0),
            #colsample_bytree=trial.suggest_float(prefix+"colsample", 0.6, 1.0),
            #reg_lambda= 1.5 #trial.suggest_float(prefix+"lambda", 1e-3, 10, log=True),
            #reg_alpha= 2.0 #trial.suggest_float(prefix+"alpha", 1e-3, 10, log=True),
            use_label_encoder=False,
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )


# ============================
# SETFIT
# ============================
def train_val(df,num_samples,labels,val=0):
        df=df.sample(frac=1)
        train_df=pd.DataFrame()
        val_df=pd.DataFrame()
        for label in labels:
            train_df=pd.concat((train_df,df[df.label==label].iloc[:(num_samples)]))
            if val:
                val_df=pd.concat((val_df,df[df.label==label].iloc[num_samples:int(num_samples*1.5)]))
            else:
                val_df=pd.concat((val_df,df[df.label==label].iloc[num_samples:]))
        return train_df,val_df

def train_setfit(df,label_col,n_val):

    model = SetFitModel.from_pretrained(MODEL_PATH)
    sgkf = StratifiedGroupKFold(n_splits=2)
    for tr_idx, val_idx in sgkf.split(df, df["has_ps"], df["pseudo_id"]):
        df_train,_=train_val(df.iloc[tr_idx],20,df["label"].iloc[tr_idx].unique(),val=1)
        df_val,_=train_val(df.iloc[val_idx],n_val,df["label"].iloc[val_idx].unique(),val=1)
        
    train_dataset = Dataset.from_pandas(
        df_train[["text", label_col]].rename(columns={label_col:"label"})
    )
    val_dataset = Dataset.from_pandas(
        df_val[["text", label_col]].rename(columns={label_col:"label"})
    )

    trainer = Trainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        args=args,
 #       learning_rate=2e-5, #, 5e-5, log=True),
        metric='f1',
        callbacks=[EarlyStoppingCallback(early_stopping_patience=4)]) #change patience to 3
    
    
    trainer.train()
    return model


# ============================
# OPTUNA OBJECTIVE
# ============================
def objective(trial, df_train, X_embed_train, X_setfit_train, X_setfit_train_mc, feat=[], mod=[], feat_mc=[], mod_mc=[]):

    inner_splits = inner_splits_all[str(fold)]


    scores = []
    precisions = []
    recalls = []
    f1s = []
    if len(feat)==0:
        feature_type = trial.suggest_categorical(
            "feature", ["tfidf","st","hybrid","setfit"]
        )
        feature_type_mc = trial.suggest_categorical(
            "feature_mc", ["tfidf","st","hybrid","setfit"]
        )
    else:
        feature_type=feat
        feature_type_mc=feat_mc

    for split in inner_splits:
        tr_idx = split["train_idx"]
        val_idx = split["val_idx"]

        df_tr = df_train.iloc[tr_idx]
        df_val = df_train.iloc[val_idx]

        #tr_idx = df_train.loc[tr_idx]
        #val_idx = df_train.loc[val_idx]

        y_bin_tr = (df_tr["label"] >= 0).astype(int)
        y_bin_val = (df_val["label"] >= 0).astype(int)

        df_tr_mc = df_tr[df_tr["label"] >= 0]
        df_val_mc = df_val[df_val["label"] >= 0]

        
        # =========================
        # FEATURES
        # =========================
        if feature_type == "tfidf":
            tfidf = TfidfVectorizer(stop_words=stopwords.words('dutch'), min_df=0.05)
            X_tr = tfidf.fit_transform(df_tr["text"])
            X_val_f = tfidf.transform(df_val["text"])
            X_tr_mc = X_tr[y_bin_tr==1]
            X_val_mc = X_val_f[y_bin_val==1]

        elif feature_type == "st":
            X_tr = X_embed_train[tr_idx]
            X_val_f = X_embed_train[val_idx]
            X_tr_mc = X_tr[y_bin_tr==1]
            X_val_mc = X_val_f[y_bin_val==1]

        elif feature_type == "setfit":
            
            X_tr= X_setfit_train[tr_idx] 
            X_tr_mc=X_setfit_train_mc[tr_idx][y_bin_tr==1]
            X_val_f= X_setfit_train[val_idx] 
            X_val_mc=X_setfit_train_mc[val_idx][y_bin_val==1]

        else:  # hybrid
            tfidf = tfidf = TfidfVectorizer(stop_words=stopwords.words('dutch'), min_df=0.05)
            X_tfidf = tfidf.fit(df_tr["text"]) #.toarray()
            X_tfidf = tfidf.transform(df_train["text"]).toarray()
            X_hybrid = np.hstack([X_setfit_train, X_tfidf])
            X_hybrid_mc = np.hstack([X_setfit_train_mc, X_tfidf])
            X_tr = X_hybrid[tr_idx]
            X_val_f = X_hybrid[val_idx]

            X_tr_mc = X_hybrid_mc[tr_idx][y_bin_tr==1]
            X_val_mc = X_hybrid_mc[val_idx][y_bin_val==1]

        if feature_type_mc == "tfidf":
            tfidf = TfidfVectorizer(stop_words=stopwords.words('dutch'), min_df=0.05)
            X_tr_t = tfidf.fit_transform(df_tr["text"])
            X_val_f_t = tfidf.transform(df_val["text"])
            X_tr_mc = X_tr_t[y_bin_tr==1]
            X_val_mc = X_val_f_t[y_bin_val==1]

        elif feature_type_mc == "st":
            X_tr_t = X_embed_train[tr_idx]
            X_val_f_t = X_embed_train[val_idx]
            X_tr_mc = X_tr_t[y_bin_tr==1]
            X_val_mc = X_val_f_t[y_bin_val==1]

        elif feature_type_mc == "setfit":
            
            #X_tr_t= X_setfit_train[tr_idx] 
            X_tr_mc=X_setfit_train_mc[tr_idx][y_bin_tr==1]
            X_val_f_t= X_setfit_train_mc[val_idx] 
            X_val_mc=X_setfit_train_mc[val_idx][y_bin_val==1]

        else:  # hybrid
            tfidf = tfidf = TfidfVectorizer(stop_words=stopwords.words('dutch'), min_df=0.05)
            X_tfidf = tfidf.fit(df_tr["text"]) #.toarray()
            X_tfidf = tfidf.transform(df_train["text"]).toarray()
            X_hybrid_mc = np.hstack([X_setfit_train_mc, X_tfidf])
            X_val_f_t = X_hybrid_mc[tr_idx]
            X_tr_mc = X_hybrid_mc[tr_idx][y_bin_tr==1]
            X_val_mc = X_hybrid_mc[val_idx][y_bin_val==1]


        # =========================
        # MODELS
        # =========================
        if len(mod)==0:
            bin_model = get_model(trial, "bin_")
            mc_model = get_model(trial, "mc_")
        else:
            bin_model =  get_model_full(trial, mod, "bin_")
            mc_model = get_model_full(trial, mod_mc, "mc_")

        bin_model.fit(X_tr, y_bin_tr)
        mc_model.fit(X_tr_mc, df_tr_mc["label"])

        # Threshold tuning
        if hasattr(bin_model, "predict_proba"):
            probs_val = bin_model.predict_proba(X_val_f)[:,1]
            threshold = find_best_threshold(y_bin_val, probs_val)
        else:
            threshold = None
        threshold=trial.suggest_float("threshold", 0.1, 0.9)

        preds = cascade_predict(bin_model, mc_model, X_val_f, X_val_f_t, threshold)

        scores.append(-rmse(df_val["label"], preds))
        precision, recall, f1, _ = precision_recall_fscore_support(df_val["label"], 
                                                           preds, average="weighted") #"macro") #
        #precisions.append(precision)
        #recalls.append(recalls)
        f1s.append(f1)

    return np.mean(f1), np.mean(scores) #, np.mean(recall), np.mean(precision)

def objective_binary(trial, df_train, X_embed_train, X_setfit_train, X_setfit_train_mc, feat=[], mod=[], feat_mc=[], mod_mc=[]):

    inner_splits = inner_splits_all[str(fold)]


    #sgkf = StratifiedGroupKFold(n_splits=N_INNER)
        

    scores = []
    precisions = []
    recalls = []
    f1s = []
    if len(feat)==0:
        feature_type = trial.suggest_categorical(
            "feature", ["tfidf","st","hybrid","setfit"]
        )
        feature_type_mc = trial.suggest_categorical(
            "feature_mc", ["tfidf","st","hybrid","setfit"]
        )
    else:
        feature_type=feat
        feature_type_mc=feat_mc

    #for tr_idx, val_idx in sgkf.split(df_train, df_train["has_ps"], df_train["pseudo_id"]):
    for split in inner_splits:
        tr_idx = split["train_idx"]
        val_idx = split["val_idx"]

        df_tr = df_train.iloc[tr_idx]
        df_val = df_train.iloc[val_idx]

        #tr_idx = df_train.loc[tr_idx]
        #val_idx = df_train.loc[val_idx]

        y_bin_tr = (df_tr["label"] >= 0).astype(int)
        y_bin_val = (df_val["label"] >= 0).astype(int)

        df_tr_mc = df_tr[df_tr["label"] >= 0]
        df_val_mc = df_val[df_val["label"] >= 0]

        
        # =========================
        # FEATURES
        # =========================
        if feature_type == "tfidf":
            tfidf = TfidfVectorizer(stop_words=stopwords.words('dutch'), min_df=0.05)
            X_tr = tfidf.fit_transform(df_tr["text"])
            X_val_f = tfidf.transform(df_val["text"])
            X_tr_mc = X_tr[y_bin_tr==1]
            X_val_mc = X_val_f[y_bin_val==1]

        elif feature_type == "st":
            X_tr = X_embed_train[tr_idx]
            X_val_f = X_embed_train[val_idx]
            X_tr_mc = X_tr[y_bin_tr==1]
            X_val_mc = X_val_f[y_bin_val==1]

        elif feature_type == "setfit":
            
            X_tr= X_setfit_train[tr_idx] 
            X_tr_mc=X_setfit_train_mc[tr_idx][y_bin_tr==1]
            X_val_f= X_setfit_train[val_idx] 
            X_val_mc=X_setfit_train_mc[val_idx][y_bin_val==1]

        else:  # hybrid
            tfidf = tfidf = TfidfVectorizer(stop_words=stopwords.words('dutch'), min_df=0.05)
            X_tfidf = tfidf.fit(df_tr["text"]) #.toarray()
            X_tfidf = tfidf.transform(df_train["text"]).toarray()
            X_hybrid = np.hstack([X_setfit_train, X_tfidf])
            X_hybrid_mc = np.hstack([X_setfit_train_mc, X_tfidf])
            X_tr = X_hybrid[tr_idx]
            X_val_f = X_hybrid[val_idx]

            X_tr_mc = X_hybrid_mc[tr_idx][y_bin_tr==1]
            X_val_mc = X_hybrid_mc[val_idx][y_bin_val==1]

        if feature_type_mc == "tfidf":
            tfidf = TfidfVectorizer(stop_words=stopwords.words('dutch'), min_df=0.05)
            X_tr_t = tfidf.fit_transform(df_tr["text"])
            X_val_f_t = tfidf.transform(df_val["text"])
            X_tr_mc = X_tr_t[y_bin_tr==1]
            X_val_mc = X_val_f_t[y_bin_val==1]

        elif feature_type_mc == "st":
            X_tr_t = X_embed_train[tr_idx]
            X_val_f_t = X_embed_train[val_idx]
            X_tr_mc = X_tr_t[y_bin_tr==1]
            X_val_mc = X_val_f_t[y_bin_val==1]

        elif feature_type_mc == "setfit":
            
            #X_tr_t= X_setfit_train[tr_idx] 
            X_tr_mc=X_setfit_train_mc[tr_idx][y_bin_tr==1]
            X_val_f_t= X_setfit_train_mc[val_idx] 
            X_val_mc=X_setfit_train_mc[val_idx][y_bin_val==1]

        else:  # hybrid
            tfidf = tfidf = TfidfVectorizer(stop_words=stopwords.words('dutch'), min_df=0.05)
            X_tfidf = tfidf.fit(df_tr["text"]) #.toarray()
            X_tfidf = tfidf.transform(df_train["text"]).toarray()
            X_hybrid = np.hstack([X_setfit_train, X_tfidf])
            X_hybrid_mc = np.hstack([X_setfit_train_mc, X_tfidf])
            X_val_f_t = X_hybrid_mc[tr_idx]
            X_tr_mc = X_hybrid_mc[tr_idx][y_bin_tr==1]
            X_val_mc = X_hybrid_mc[val_idx][y_bin_val==1]


        # =========================
        # MODELS
        # =========================
        if len(mod)==0:
            bin_model = get_model(trial, "bin_")
            mc_model = get_model(trial, "mc_")
        else:
            bin_model =  get_model_full(trial, mod, "bin_")
            mc_model = get_model_basic(trial, mod_mc, "mc_")

        bin_model.fit(X_tr, y_bin_tr)
        mc_model.fit(X_tr_mc, df_tr_mc["label"])

        # Threshold tuning
        #if hasattr(bin_model, "predict_proba"):
        #    probs_val = bin_model.predict_proba(X_val_f)[:,1]
        #    threshold = find_best_threshold(y_bin_val, probs_val)
        #else:
        #    threshold = None
        threshold=trial.suggest_float("threshold", 0.1, 0.9)

        preds = cascade_predict(bin_model, mc_model, X_val_f, X_val_f_t, threshold)
        preds_binary= [0 if preds[i]==-1 else 1 for i in range(len(preds))]
        scores.append(-rmse(y_bin_val, preds_binary))
        precision, recall, f1, _ = precision_recall_fscore_support(y_bin_val, 
                                                           preds_binary, average="weighted") #"macro") #
        #precisions.append(precision)
        #recalls.append(recalls)
        f1s.append(f1)

    return np.mean(f1) #, np.mean(recall), np.mean(precision)

def objective_mc(trial, df_train, X_embed_train, X_setfit_train, X_setfit_train_mc, bin_model, feat, feat_mc, mod_mc, threshold):

    inner_splits = inner_splits_all[str(fold)]


    #sgkf = StratifiedGroupKFold(n_splits=N_INNER)
        

    scores = []
    
    f1s = []

    feature_type=feat
    feature_type_mc=feat_mc

     for split in inner_splits:
        tr_idx = split["train_idx"]
        val_idx = split["val_idx"]

        df_tr = df_train.iloc[tr_idx]
        df_val = df_train.iloc[val_idx]

        y_bin_tr = (df_tr["label"] >= 0).astype(int)
        y_bin_val = (df_val["label"] >= 0).astype(int)

        df_tr_mc = df_tr[df_tr["label"] >= 0]
        df_val_mc = df_val[df_val["label"] >= 0]

        
        # =========================
        # FEATURES
        # =========================
        if feature_type == "tfidf":
            tfidf = TfidfVectorizer(stop_words=stopwords.words('dutch'), min_df=0.05)
            X_tr = tfidf.fit_transform(df_tr["text"])
            X_val_f = tfidf.transform(df_val["text"])
            X_tr_mc = X_tr[y_bin_tr==1]
            X_val_mc = X_val_f[y_bin_val==1]

        elif feature_type == "st":
            X_tr = X_embed_train[tr_idx]
            X_val_f = X_embed_train[val_idx]
            X_tr_mc = X_tr[y_bin_tr==1]
            X_val_mc = X_val_f[y_bin_val==1]

        elif feature_type == "setfit":
            
            X_tr= X_setfit_train[tr_idx] 
            X_tr_mc=X_setfit_train_mc[tr_idx][y_bin_tr==1]
            X_val_f= X_setfit_train[val_idx] 
            X_val_mc=X_setfit_train_mc[val_idx][y_bin_val==1]

        else:  # hybrid
            tfidf = tfidf = TfidfVectorizer(stop_words=stopwords.words('dutch'), min_df=0.05)
            X_tfidf = tfidf.fit(df_tr["text"]) #.toarray()
            X_tfidf = tfidf.transform(df_train["text"]).toarray()
            X_hybrid = np.hstack([X_setfit_train, X_tfidf])
            X_hybrid_mc = np.hstack([X_setfit_train_mc, X_tfidf])
            X_tr = X_hybrid[tr_idx]
            X_val_f = X_hybrid[val_idx]

            X_tr_mc = X_hybrid_mc[tr_idx][y_bin_tr==1]
            X_val_mc = X_hybrid_mc[val_idx][y_bin_val==1]

        if feature_type_mc == "tfidf":
            tfidf = TfidfVectorizer(stop_words=stopwords.words('dutch'), min_df=0.05)
            X_tr_t = tfidf.fit_transform(df_tr["text"])
            X_val_f_t = tfidf.transform(df_val["text"])
            X_tr_mc = X_tr_t[y_bin_tr==1]
            X_val_mc = X_val_f_t[y_bin_val==1]

        elif feature_type_mc == "st":
            X_tr_t = X_embed_train[tr_idx]
            X_val_f_t = X_embed_train[val_idx]
            X_tr_mc = X_tr_t[y_bin_tr==1]
            X_val_mc = X_val_f_t[y_bin_val==1]

        elif feature_type_mc == "setfit":
            
            #X_tr_t= X_setfit_train[tr_idx] 
            X_tr_mc=X_setfit_train_mc[tr_idx][y_bin_tr==1]
            X_val_f_t= X_setfit_train_mc[val_idx] 
            X_val_mc=X_setfit_train_mc[val_idx][y_bin_val==1]

        else:  # hybrid
            tfidf = tfidf = TfidfVectorizer(stop_words=stopwords.words('dutch'), min_df=0.05)
            X_tfidf = tfidf.fit(df_tr["text"]) #.toarray()
            X_tfidf = tfidf.transform(df_train["text"]).toarray()
            X_hybrid = np.hstack([X_setfit_train, X_tfidf])
            X_hybrid_mc = np.hstack([X_setfit_train_mc, X_tfidf])
            X_val_f_t = X_hybrid_mc[tr_idx]
            X_tr_mc = X_hybrid_mc[tr_idx][y_bin_tr==1]
            X_val_mc = X_hybrid_mc[val_idx][y_bin_val==1]


        # =========================
        # MODELS
        # =========================

        
        mc_model = get_model_full(trial, mod_mc, "mc_")

        bin_model.fit(X_tr, y_bin_tr)
        mc_model.fit(X_tr_mc, df_tr_mc["label"])


        preds = mc_model.predict(X_val_mc)

        scores.append(-rmse(df_val["label"][y_bin_val==1], preds))
        precision, recall, f1, _ = precision_recall_fscore_support(df_val["label"][y_bin_val==1], 
                                                           preds, average="weighted") #"macro") #
        #precisions.append(precision)
        #recalls.append(recalls)
        f1s.append(f1)

    return np.mean(scores) #np.mean(f1),  #, np.mean(recall), np.mean(precision)


def get_model_full(trial, model_type, prefix=""): 



    if model_type == "lr":
        return LogisticRegression(
            C=trial.suggest_float(prefix+"C", 1e-2, 10, log=True),
            max_iter=1000
        )

    elif model_type == "rf":
        return RandomForestClassifier(
            n_estimators=trial.suggest_int(prefix+"n", 50, 200),
            max_depth=trial.suggest_int(prefix+"depth", 3, 10),
            random_state=RANDOM_STATE
        )

    else:
        return XGBClassifier(
            #n_estimators=trial.suggest_int(prefix+"n", 50, 300),
            max_depth=trial.suggest_int(prefix+"depth", 3, 10),
            learning_rate=trial.suggest_float(prefix+"lr", 0.01, 0.3, log=True),
            subsample=trial.suggest_float(prefix+"subsample", 0.6, 1.0),
            colsample_bytree=trial.suggest_float(prefix+"colsample", 0.6, 1.0),
            reg_lambda= trial.suggest_float(prefix+"lambda", 1e-3, 10, log=True), #1.5, #
            reg_alpha= trial.suggest_float(prefix+"alpha", 1e-3, 10, log=True), #2.0, #
            use_label_encoder=False,
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )


def get_features(feature_type,feature_type_mc):
    if feature_type == "tfidf":
        tfidf = TfidfVectorizer(stop_words=stopwords.words('dutch'), min_df=0.05)
        X_train = tfidf.fit_transform(df_train["text"])
        X_test = tfidf.transform(df_test["text"])
        

    elif feature_type == "st":
        X_train = X_embed_train
        X_test = X_embed_test
        
    elif feature_type == "setfit":
        X_train=X_setfit_train
        X_test=X_setfit_test
        
        
    else:
        tfidf = TfidfVectorizer(stop_words=stopwords.words('dutch'), min_df=0.05)
        X_tfidf = tfidf.fit(df_train["text"])#toarray()
        X_tfidf = tfidf.transform(df["text"]).toarray()
        X_hybrid = np.hstack([X_sf_bin, X_tfidf])
        X_train = X_hybrid[train_idx]
        X_test = X_hybrid[test_idx]
        

    if feature_type_mc == "tfidf":
        tfidf = TfidfVectorizer(stop_words=stopwords.words('dutch'), min_df=0.05)
        X_train_mc = tfidf.fit_transform(df_train["text"])[y_bin_train==1]
        X_test_mc = tfidf.transform(df_test["text"])[y_bin_test==1]
        X_test_t = tfidf.transform(df_test["text"])

    elif feature_type_mc == "st":
        X_train_mc = X_embed_train[y_bin_train==1]
        X_test_mc = X_embed_test[y_bin_test==1]
        X_test_t = X_embed_test
        
    elif feature_type_mc == "setfit":

        X_train_mc =  X_setfit_train_mc[y_bin_train==1]
        X_test_mc =  X_setfit_test_mc[y_bin_test==1]
        X_test_t =  X_setfit_test_mc
        
    else:
        tfidf = TfidfVectorizer(stop_words=stopwords.words('dutch'), min_df=0.05)
        X_tfidf = tfidf.fit(df_train["text"])#toarray()
        X_tfidf = tfidf.transform(df["text"]).toarray()
        X_hybrid_mc = np.hstack([X_sf_mc, X_tfidf])
        X_train_mc =X_hybrid_mc[train_idx][y_bin_train==1]
        X_test_mc = X_hybrid_mc[test_idx][y_bin_test==1]  
        X_test_t = X_hybrid_mc[test_idx]
    
    
    return X_train,X_train_mc,X_test,X_test_t

def create_table(results_combi):
    index_name=[]
    final_results_cv=pd.DataFrame()
    feature_types=["tfidf","st","hybrid","setfit"]
    model_types=["lr","xgb"]
    # Run Optuna for all combinations
    for feat, mod in product(feature_types, model_types): 
        for feat_mc, mod_mc in product(feature_types, model_types): 
            results=[]
            for i in range(len(results_combi)):
                if results_combi[i]['feature']==feat and results_combi[i]['model']==mod and results_combi[i]['model multiclass']==mod_mc and results_combi[i]['feature multiclass']==feat_mc:
                    results.append(results_combi[i])
                    index_name=('Embedding type: ' + str(feat) + ' + model:  ' + str(mod) + ' Embedding type mc: ' + str(feat_mc) + ' + model mc:  ' + str(mod_mc))
                
            prec=np.mean([results[i]['precision'] for i in range(len(results))])
            recall=np.mean([results[i]['recall'] for i in range(len(results))])
            f1=np.mean([results[i]['f1'] for i in range(len(results))])
            adj_acc=np.mean([results[i]['adjacent accuracy'] for i in range(len(results))])
            rmse=np.mean([results[i]['rmse'] for i in range(len(results))])
            
            prec_sd=np.std([results[i]['precision'] for i in range(len(results))])
            recall_sd=np.std([results[i]['recall'] for i in range(len(results))])
            f1_sd=np.std([results[i]['f1'] for i in range(len(results))])
            adj_acc_sd=np.std([results[i]['adjacent accuracy'] for i in range(len(results))])
            rmse_sd=np.std([results[i]['rmse'] for i in range(len(results))])

        
        
            final_results_cv.loc[index_name,['Precision', 'Precision sd', 'Recall', 'Recall sd', 
                                     'F1 score', 'F1 score sd', 'RMSE', 'RMSE sd', 
                                     'Adjacent accuracy', 'Adjacent accuracy sd']]=[prec, prec_sd,recall, recall_sd, f1, f1_sd, rmse, rmse_sd, adj_acc, adj_acc_sd]     
            for col in "recall binary","precision binary","f1 binary","precision multiclass","recall multiclass","f1 multiclass", "rmse multiclass":
            
               final_results_cv.loc[index_name,col]=np.mean([results[i][col] for i in range(len(results))])
               final_results_cv.loc[index_name,col + 'sd']=np.sd([results[i][col] for i in range(len(results))])
    return final_results_cv 

def predict(results,df, bin_model, mc_model,feature=None, feature_mc=None,threshold=None):
    print(f"\n===== FULL MODEL =====")
    best_params =  best_trial.params
    print("Best params:", best_params)
    
    if feature:
        feature_type=feature
        feature_type_mc=feature_mc
    else:
        feature_type =results["feature binary"]
        feature_type_mc =results["feature multiclass"]
    if feature_type=='st':
        print("Computing embeddings...")
        embed_model = SentenceTransformer(MODEL_PATH)
        df["embedding"] = list(embed_model.encode(df["text"].tolist()))
        X_embed_all = np.vstack(df["embedding"].values)
    
    if threshold:
        threshold=threshold
    else:
        threshold=results["threshold"]
    sf_bin_path = os.path.join(SAVE_DIR, f"allnotes_setfit_bin.npy")
    sf_mc_path = os.path.join(SAVE_DIR, f"allnotes_setfit_mc.npy")
    
    X_sf_bin = get_setfit_embeddings(bin_model_setfit, df["pseudonomised_text"].tolist(), sf_bin_path)
    X_sf_mc = get_setfit_embeddings(mc_model_setfit, df["pseudonomised_text"].tolist(), sf_mc_path)
        
    
    if feature_type == "tfidf":
        tfidf = TfidfVectorizer(stop_words=stopwords.words('dutch'), min_df=0.05)
        X= tfidf.fit(df_train["text"])
        X= tfidf.transform(df["pseudonomised_text"])
    
    
    elif feature_type == "st":
        X = X_embed_all
    
        
    elif feature_type == "setfit":
        X = X_sf_bin
    
      
    else:
        tfidf = TfidfVectorizer(stop_words=stopwords.words('dutch'), min_df=0.05)
        X_tfidf= tfidf.fit(df_train["text"])
        X_tfidf= tfidf.transform(df["pseudonomised_text"]).toarray()
        X = np.hstack([X_sf_bin, X_tfidf])
    
    
    if feature_type_mc == "tfidf":
        tfidf = TfidfVectorizer(stop_words=stopwords.words('dutch'), min_df=0.05)
        X_mc= tfidf.fit(df_train["text"])
        X_mc= tfidf.transform(df["pseudonomised_text"])
    
    elif feature_type_mc == "st":
        X_mc=X_embed_all
        
    elif feature_type_mc == "setfit":
        X_mc = X_sf_mc
      
    else:
        tfidf = TfidfVectorizer(stop_words=stopwords.words('dutch'), min_df=0.05)
        X_tfidf= tfidf.fit(df_train["text"])
        X_tfidf= tfidf.transform(df["pseudonomised_text"]).toarray()
        X_mc = np.hstack([X_sf_mc, X_tfidf])
    
    
    print(bin_model)
    print(X)
    print(np.shape(X))  
    print(np.shape(X_mc)) 
    
    preds = cascade_predict(bin_model, mc_model, X, X_mc, threshold)
    
    
    return preds


In [ ]:
N_TRIALS=60

SAVE_DIR="Z:\PS_classification\Experiment2"
OUTER_FOLD_PATH = os.path.join(SAVE_DIR, "outer_folds.json")

results_all_combi=[]
if os.path.exists(OUTER_FOLD_PATH):
    with open(OUTER_FOLD_PATH, "r") as f:
        outer_splits = json.load(f)
else:
    outer_splits = []
    sgkf_outer = sgkf_outer = StratifiedGroupKFold(n_splits=N_OUTER)
    for fold, (train_idx, test_idx) in enumerate(
            sgkf_outer.split(df, df["label"], df["pseudo_id"])
        ):
            outer_splits.append({
                "fold": fold,
                "train_idx": train_idx.tolist(),
                "test_idx": test_idx.tolist()
            })

    with open(OUTER_FOLD_PATH, "w") as f:
        json.dump(outer_splits, f)

INNER_FOLD_PATH = os.path.join(SAVE_DIR, "inner_folds.json")

if os.path.exists(INNER_FOLD_PATH):
    with open(INNER_FOLD_PATH, "r") as f:
        inner_splits_all = json.load(f)
else:
    inner_splits_all = {}

    for outer in outer_splits:
        fold = outer["fold"]
        train_idx = outer["train_idx"]

        df_train = df.iloc[train_idx]

        sgkf_inner = StratifiedGroupKFold(
            n_splits=N_INNER
        )

        inner_splits = []
        for inner_fold, (tr_idx, val_idx) in enumerate(
            sgkf_inner.split(df_train, df_train["label"], df_train["pseudo_id"])
        ):
            inner_splits.append({
                "inner_fold": inner_fold,
                "train_idx": tr_idx.tolist(), #df_train.index[tr_idx].tolist(),
                "val_idx": val_idx.tolist() #df_train.index[val_idx].tolist()
            })

        inner_splits_all[str(fold)] = inner_splits

    with open(INNER_FOLD_PATH, "w") as f:
        json.dump(inner_splits_all, f)




fold_nr=0
#for fold, (train_idx, test_idx) in enumerate(outer_splits): #sgkf_outer.split(df, df["label"], df["pseudo_id"])):
for outer_split in outer_splits:
    train_idx = outer_split["train_idx"]
    test_idx = outer_split["test_idx"]
    fold = outer_split["fold"]
    print(f"\n===== FOLD {fold} =====")
       
    df_train = df.iloc[train_idx]
    df_test = df.iloc[test_idx]

    X_embed_train = X_embed_all[train_idx]
    X_embed_test = X_embed_all[test_idx]

    y_bin_train = (df_train["label"]>=0).astype(int)
    y_bin_test = (df_test["label"]>=0).astype(int)

    df_train_mc = df_train[df_train["label"]>=0]
    df_test_mc= df_test[df_test["label"]>=0]

    sf_bin_path = os.path.join(SAVE_DIR, f"fold_{fold}_setfit_bin.npy")
    sf_mc_path = os.path.join(SAVE_DIR, f"fold_{fold}_setfit_mc.npy")
    
    if os.path.exists(sf_bin_path)==False:
        bin_model_setfit = train_setfit(df_train.assign(label=y_bin_train),"label",20)
    else:
        bin_model_setfit=[]
        
    if os.path.exists(sf_mc_path)==False:
        mc_model_setfit = train_setfit(df_train_mc,"label",8)
    else:
        mc_model_setfit=[]
        
    X_sf_bin= get_setfit_embeddings(bin_model_setfit, df["text"].tolist(), sf_bin_path)
    X_sf_mc = get_setfit_embeddings(mc_model_setfit, df["text"].tolist(), sf_mc_path)
    

    X_setfit_train = X_sf_bin[train_idx] #bin_model_setfit.model_body.encode(df_train["text"].values.tolist())
    X_setfit_train_mc = X_sf_mc[train_idx] #mc_model_setfit.model_body.encode(df_train["text"].values.tolist())
    X_setfit_test = X_sf_bin[test_idx] #bin_model_setfit.model_body.encode(df_test["text"].values.tolist())
    X_setfit_test_mc= X_sf_mc[test_idx] #mc_model_setfit.model_body.encode(df_test["text"].values.tolist())
        # =========================

    results_fold=[]
    best_trial_full=[]
    threshold_full=[]
    best_trial_full_mc=[]
    from itertools import product
    # Run Optuna for all combinations

    for feat, mod in product(feature_types, model_types):
        study_full = optuna.create_study(directions=["maximize"]) #,"maximize"]) #,"maximize","maximize"])
        study_full.optimize(lambda trial: objective_binary(trial,df_train,X_embed_train,X_setfit_train,X_setfit_train_mc, feat, mod, feat_mc, mod_mc), n_trials=N_TRIALS)
        #fig = optuna.visualization.plot_optimization_history(study_full, target=0)
        #show(fig)
        best_trial_full.append(max(study_full.best_trials, key=lambda t: t.values[0])) #best f1 score (1: RMSE, 2: recall, 3: precision)
        threshold_full.append(best_trial_full[-1].params["threshold"])

    bin_model_full=get_model_full(best_trial_full[0],'lr', "bin_")
    for feat_mc, mod_mc in product(feature_types, model_types):
            study_full_mc = optuna.create_study(directions=["maximize"]) #,"maximize"]) #,"maximize","maximize"])
            study_full_mc.optimize(lambda trial: objective_mc(trial,df_train,X_embed_train,X_setfit_train,X_setfit_train_mc, bin_model_full, feat, feat_mc, mod_mc, threshold_full[0]), n_trials=N_TRIALS)
            #fig = optuna.visualization.plot_optimization_history(study_full, target=0)
            #show(fig)
            best_trial_full_mc.append(max(study_full_mc.best_trials, key=lambda t: t.values[0])) #best f1 score (1: RMSE, 2: recall, 3: precision))
   
    i=0
    for feat, mod in product(feature_types, model_types):
        j=0
        bin_model_full = get_model_full(best_trial_full[i],mod, "bin_")
        
        
        
        for feat_mc, mod_mc in product(feature_types, model_types):
            X_train,X_train_mc,X_test,X_test_t=get_features(feat,feat_mc)
            mc_model_full = get_model_full(best_trial_full_mc[j],mod_mc, "mc_")
            #bin_model_full = get_model_retrain(best_trial_full.params['bin_model'], best_params)
            #mc_model_full = get_model_retrain(best_trial_full.params['bin_model'], best_params)
            bin_model_full.fit(X_train, y_bin_train)
            mc_model_full.fit(X_train_mc, df_train_mc["label"])
            
            
            
            preds_full = cascade_predict(bin_model_full, mc_model_full, X_test, X_test_t, threshold_full[i])
            precision, recall, f1, _ = precision_recall_fscore_support(df_test["label"], 
                                                                       preds_full, average="weighted")
            
            bin_labels=[0 if df_test["label"].iloc[i]==-1 else 1 for i in range(len(df_test))]
            bin_labels_pred=[0 if preds_full[i]==-1 else 1 for i in range(len(preds_full))]
            precision_b, recall_b, f1_b, _ = precision_recall_fscore_support(bin_labels, 
                                                                       bin_labels_pred, average="weighted")
            
            sub_class_metrics_b = precision_recall_fscore_support(bin_labels, 
                                                                       bin_labels_pred)
            sub_class_metrics = precision_recall_fscore_support(df_test["label"], 
                                                                       preds_full)
            subfold_rmse= rmse(df_test["label"], preds_full)
            pred_full_adj=[pred_i if pred_i>-1 else -2 for pred_i in preds_full]
            subfold_custom_adjacent_accuracy = custom_adjacent_accuracy(df_test["label"].replace(-1,-2), pred_full_adj)
             
            print(f"Fold {fold}, Feature {feat}, Model {mod},Feature multiclass {feat_mc}, Model multiclass {mod_mc}, Best RMSE test set: {subfold_rmse:4f},Best F1 score test set: {f1:.4f}")
            precision_mc, recall_mc, f1_mc, _ = precision_recall_fscore_support(df_test["label"][df_test["label"]>=0], preds_full[df_test["label"]>=0], average="weighted")
            
    
            sub_class_metrics_mc = precision_recall_fscore_support(df_test["label"][df_test["label"]>=0], 
                                                                       preds_full[df_test["label"]>=0])
            subfold_rmse_mc= rmse(df_test["label"][df_test["label"]>=0], preds_full[df_test["label"]>=0])
            

            results_fold.append({"feature":feat,
                "model":mod,
                "feature multiclass":feat_mc,
                "model multiclass":mod_mc,
                "optuna_study binary":best_trial_full[i],
                "optuna_study":best_trial_full_mc[j],
                "fold": fold,
                "precision": precision,
                "recall": recall,
                "f1": f1,
                "precision binary": precision_b,
                "recall binary": recall_b,
                "f1 binary": f1_b,
                "precision multiclass": precision_mc,
                "recall multiclass": recall_mc,
                "f1 multiclass": f1_mc,
                "rmse multiclass": subfold_rmse_mc,
                "rmse": subfold_rmse,
                "threshold": threshold_full,
                "class_metrics": sub_class_metrics,
                "class_metrics multiclass": sub_class_metrics_mc,
                "class_metrics binary": sub_class_metrics_b,
                "adjacent accuracy": subfold_custom_adjacent_accuracy,
                #"model binary": best_trial_full.params['bin_model'], 
                #"model multiclass": best_trial_full.params['mc_model'],
                "Params binary": best_trial_full[i].params,
                "Params multiclass": best_trial_full_mc[j].params,
                #"val precision": best_trial_full.values[3],
                #"val recall": best_trial_full.values[2],
                "val f1": best_trial_full[i].values[0],
                "val rmse": best_trial_full_mc[j].values[0],       
                #"Type feature multiclass": feat_type_multiclass
            })
                
            j=j+1
        # After inner loop, train best hyperparams on full training fold and evaluate on test
        i=i+1
    best_result = min(results_fold, key=lambda x:x["rmse"])
        
    print(f"Fold {fold} BEST combination: RMSE={best_result['rmse']},F1={best_result['f1']}, Feature={best_result['feature']}, Model={best_result['model']}, Feature multiclass={best_result['feature multiclass']}, Model multiclass={best_result['model multiclass']}")
    results_all_combi.append(results_fold)
        

    fold_nr=fold_nr+1
 
    np.save('Thresholds_fold+' + str(fold_nr) +'.npy',threshold_full)
    np.save('Trial_binary_fold_' + str(fold_nr) +'.npy',best_trial_full)   
    np.save('Trial_multiclass_fold_' + str(fold_nr) +'.npy',best_trial_full_mc)
np.save('Results_full_combi_cv_paper_v2.npy',np.hstack(results_all_combi))
results_combi=np.hstack(results_all_combi)
final_table=create_table(results_combi)
final_table.sort_values(by=['RMSE'])
final_table.to_excel('Z:\PS_classification\Results_paper\Final_results_RMSE_v2.xlsx')

### Train full model with best combination of features and model types

In [ ]:
SAVE_DIR="Z:\PS_classification\Experiment_full2"
N_TRIALS=100
#Save full model

# ============================
# OUTER LOOP
# ============================

OUTER_FOLD_PATH = os.path.join(SAVE_DIR, "outer_folds_full.json")

if os.path.exists(OUTER_FOLD_PATH):
    with open(OUTER_FOLD_PATH, "r") as f:
        outer_splits = json.load(f)
else:
    outer_splits = []
    fold=0
    outer_splits.append({
        "fold": fold,
        "train_idx":df.index.tolist(),
        "test_idx": df.index.tolist()
    })

    with open(OUTER_FOLD_PATH, "w") as f:
        json.dump(outer_splits, f)

INNER_FOLD_PATH = os.path.join(SAVE_DIR, "inner_folds_full.json")

if os.path.exists(INNER_FOLD_PATH):
    with open(INNER_FOLD_PATH, "r") as f:
        inner_splits_all = json.load(f)
else:
    inner_splits_all = {}

    for outer in outer_splits:
        fold = outer["fold"]
        train_idx = outer["train_idx"]

        df_train = df.iloc[train_idx]

        sgkf_inner = StratifiedGroupKFold(
            n_splits=N_INNER
        )

        inner_splits = []
        for inner_fold, (tr_idx, val_idx) in enumerate(
            sgkf_inner.split(df_train, df_train["label"], df_train["pseudo_id"])
        ):
            inner_splits.append({
                "inner_fold": inner_fold,
                "train_idx": tr_idx.tolist(), #df_train.index[tr_idx].tolist(),
                "val_idx": val_idx.tolist() #df_train.index[val_idx].tolist()
            })

        inner_splits_all[str(fold)] = inner_splits

    with open(INNER_FOLD_PATH, "w") as f:
        json.dump(inner_splits_all, f)



results = []

#for fold, (train_idx, test_idx) in enumerate(outer_splits): #sgkf_outer.split(df, df["label"], df["pseudo_id"])):
for outer_split in outer_splits:
    train_idx = outer_split["train_idx"]
    test_idx = outer_split["test_idx"]
    fold = outer_split["fold"]
    print(f"\n===== FULL MODEL =====")
       
    df_train = df.iloc[train_idx]
    df_test = df.iloc[test_idx]

    X_embed_train = X_embed_all[train_idx]
    X_embed_test = X_embed_all[test_idx]

    y_bin_train = (df_train["label"]>=0).astype(int)
    y_bin_test = (df_test["label"]>=0).astype(int)

    df_train_mc = df_train[df_train["label"]>=0]
    df_test_mc= df_test[df_test["label"]>=0]

    sf_bin_path = os.path.join(SAVE_DIR, f"fold_{fold}_setfit_bin.npy")
    sf_mc_path = os.path.join(SAVE_DIR, f"fold_{fold}_setfit_mc.npy")
    
    if os.path.exists(sf_bin_path)==False:
        bin_model_setfit = train_setfit(df_train.assign(label=y_bin_train),"label",20)
    else:
        bin_model_setfit=None
        
    if os.path.exists(sf_mc_path)==False:
        mc_model_setfit = train_setfit(df_train_mc,"label",8)
    else:
       mc_model_setfit=None 
        
    X_sf_bin= get_setfit_embeddings(bin_model_setfit, df["text"].tolist(), sf_bin_path)
    X_sf_mc = get_setfit_embeddings(mc_model_setfit, df["text"].tolist(), sf_mc_path)
    

    X_setfit_train = X_sf_bin[train_idx] #bin_model_setfit.model_body.encode(df_train["text"].values.tolist())
    X_setfit_train_mc = X_sf_mc[train_idx] #mc_model_setfit.model_body.encode(df_train["text"].values.tolist())
    X_setfit_test = X_sf_bin[test_idx] #bin_model_setfit.model_body.encode(df_test["text"].values.tolist())
    X_setfit_test_mc= X_sf_mc[test_idx] #mc_model_setfit.model_body.encode(df_test["text"].values.tolist())
        # =========================
    feat='hybrid'
    mod='xgb'
    feat_mc='tfidf'
    mod_mc='xgb'
    study = optuna.create_study(directions=["maximize"]) #,"maximize"]) #,"maximize","maximize"])
    study.optimize(lambda trial: objective_binary(trial,df_train,X_embed_train,X_setfit_train,X_setfit_train_mc, feat, mod, feat_mc, mod_mc), n_trials=N_TRIALS)
    best_trial=max(study.best_trials, key=lambda t: t.values[0]) #best f1 score (1: RMSE, 2: recall, 3: precision)
    threshold=best_trial.params["threshold"]

    bin_model = get_model_full(best_trial,mod, "bin_")
        
    study_mc = optuna.create_study(directions=["maximize"]) #,"maximize"]) #,"maximize","maximize"])
    study_mc.optimize(lambda trial: objective_mc(trial,df_train,X_embed_train,X_setfit_train,X_setfit_train_mc, bin_model, feat, feat_mc, mod_mc, threshold_full[0]), n_trials=N_TRIALS)
    #fig = optuna.visualization.plot_optimization_history(study_full, target=0)
    #show(fig)
    best_trial_mc=max(study_mc.best_trials, key=lambda t: t.values[0]) #best f1 score (1: RMSE, 2: recall, 3: precision))
     
            
     
    X_train,X_train_mc,X_test,X_test_t=get_features(feat,feat_mc)
    mc_model = get_model_full(best_trial_mc,mod_mc, "mc_")
                
    bin_model.fit(X_train, y_bin_train)
    mc_model.fit(X_train_mc, df_train_mc["label"])
                
                
                
    preds = cascade_predict(bin_model, mc_model, X_test, X_test_t, threshold)
    precision, recall, f1, _ = precision_recall_fscore_support(df_test["label"], 
                                                                           preds, average="weighted")
    
    bin_labels=[0 if df_test["label"].iloc[i]==-1 else 1 for i in range(len(df_test))]
    bin_labels_pred=[0 if preds[i]==-1 else 1 for i in range(len(preds))]
    precision_b, recall_b, f1_b, _ = precision_recall_fscore_support(bin_labels, 
                                                               bin_labels_pred, average="weighted")
    
    sub_class_metrics_b = precision_recall_fscore_support(bin_labels, 
                                                               bin_labels_pred)
    sub_class_metrics = precision_recall_fscore_support(df_test["label"], 
                                                               preds)
    subfold_rmse= rmse(df_test["label"], preds)
    pred_full_adj=[pred_i if pred_i>-1 else -2 for pred_i in preds_full]
    subfold_custom_adjacent_accuracy = custom_adjacent_accuracy(df_test["label"].replace(-1,-2), pred_full_adj)

    print(f"Fold {fold}, Feature {feat}, Model {mod},Feature multiclass {feat_mc}, Model multiclass {mod_mc}, Best RMSE test set: {subfold_rmse:4f},Best F1 score test set: {f1:.4f}")
    precision_mc, recall_mc, f1_mc, _ = precision_recall_fscore_support(df_test["label"][df_test["label"]>=0], preds[df_test["label"]>=0], average="weighted")
    

    sub_class_metrics_mc = precision_recall_fscore_support(df_test["label"][df_test["label"]>=0], 
                                                               preds[df_test["label"]>=0])
    subfold_rmse_mc= rmse(df_test["label"][df_test["label"]>=0], preds[df_test["label"]>=0])
                

    results.append({"feature":feat,
        "model":mod,
        "feature multiclass":feat_mc,
        "model multiclass":mod_mc,
        "optuna_study binary":best_trial,
        "optuna_study":best_trial_mc,
        "fold": fold,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "precision binary": precision_b,
        "recall binary": recall_b,
        "f1 binary": f1_b,
        "precision multiclass": precision_mc,
        "recall multiclass": recall_mc,
        "f1 multiclass": f1_mc,
        "rmse multiclass": subfold_rmse_mc,
        "rmse": subfold_rmse,
        "threshold": threshold,
        "class_metrics": sub_class_metrics,
        "class_metrics multiclass": sub_class_metrics_mc,
        "class_metrics binary": sub_class_metrics_b,
        "adjacent accuracy": subfold_custom_adjacent_accuracy,
        "Params binary": best_trial.params,
        "Params multiclass": best_trial_mc.params,
        "val f1": best_trial.values[0],
        "val rmse": best_trial_mc.values[0],       
    })

    
np.save('Results_complete.npy',np.hstack(results))
SAVE_DIR="Z:\PS_classification\Experiment_full2"
joblib.dump(bin_model,os.path.join(SAVE_DIR, "bin_model.pkl"))
joblib.dump(mc_model,os.path.join(SAVE_DIR, "mc_model.pkl")) 
joblib.dump(best_trial,os.path.join(SAVE_DIR, "best_trial.pkl"))

In [ ]:
notes_PS=pd.read_csv(r'Z:\PS_classification\data_experiment\notes_classified_regex.csv')

notes_PS['PS_semantic']=predict(results,notes_PS,bin_model,mc_model,feat,feat_mc,threshold) #[notes_PS.PS=='-1']

notes_PS['Composite_PS']=[str(int(notes_PS['PS_semantic'].iloc[i])) if notes_PS['PS'].iloc[i]=='-1' else notes_PS['PS'].iloc[i] for i in range(len(notes_PS))]
notes_PS.to_csv(r'Z:\PS_classification\data_experiment\notes_classified_semantics_v2.csv')
notes_PS.PS.value_counts()

In [ ]:
plt.hist(n_meas,30)
plt.xlabel('Number of Composite PS measurements per patient')
plt.xlabel('Number of patients')
plt.savefig('Nrmeasurements_perpatient_hist.png')

In [ ]:
plt.hist(notes_PS2.idx,30)
plt.xlabel('Days from diagnosis')
plt.ylabel('Number of composite PS measurements')
plt.savefig('Nrmeasurements_hist.png')